In [2]:

from nano_model import GPT, GPTConfig
import copy
import torch
from pathlib import Path
import yaml
import pickle


In [4]:
base_dir = Path.cwd()
ckpt_path = base_dir/'model/ckpt.pt'
meta_path = base_dir/'model/meta.pkl'
print(f"Loading meta from {meta_path}...")
with open(meta_path, 'rb') as f:
    meta = pickle.load(f)

stoi, itos = meta['stoi'], meta['itos']
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
device = 'cuda'

checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
model = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)
model.to(device)
print()

Loading meta from /afs/cs.wisc.edu/u/r/i/riyad/Desktop/Foundation_model/hw3/model/meta.pkl...


/tmp/ipykernel_2681323/3652391595.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


number of parameters: 10.65M



In [6]:
from reward_model import GPT as RewardGPT
from reward_model import GPTConfig  as RewardGPTConfig
# load reward model 
base_path = Path.cwd()

reward_meta_data = base_path/ "meta.pkl"
reward_meta_data = pickle.load(open(reward_meta_data, "rb"))
with open('config/config_reward.yaml') as f:
    conf = yaml.load(f, Loader=yaml.FullLoader)
    # nested dictionary structure
    config = {}               
    for k, v in conf.items():
        for k2, v2 in v.items():
            config[k2] = v2

vocab_size = reward_meta_data['vocab_size']
reward_model_args = dict(n_layer=config['n_layer'], n_head=config['n_head'], n_embd=config['n_embd'], block_size=config['block_size'],
                    bias=config['bias'], vocab_size=vocab_size, dropout=config['dropout'], ) # start with model_args from command line
reward_model = RewardGPT(RewardGPTConfig(**reward_model_args))
reward_model_dict = torch.load(base_path/"reward_model.pth", map_location="cpu")
reward_model.load_state_dict(reward_model_dict)
reward_model = reward_model.cuda()
reward_model.eval()
#
print()

number of parameters: 3.61M



/tmp/ipykernel_2681323/855049450.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  reward_model_dict = torch.load(base_path/"reward_model.pth", map_location="cpu")


In [7]:
# sample a completion from base nano gpt and repot the mean verifier score
#a few representative examples
# samples
def generate(model,meta_data,prompt_tokens=None, max_new_tokens=100, device='cuda'):
    """
    Generate text from the model.
    
    Args:
        model: GPT model
        prompt_tokens: optional tensor of shape (1, L) with initial tokens
        max_new_tokens: number of tokens to generate
        device: cuda or cpu
    
    Returns:
        generated_tokens: tensor of shape (1, L+max_new_tokens)
    """
   
    generated = []
  
    if prompt_tokens is None:
        prompt_tokens = "\n"
    
    top_k = 200 
    stoi, itos = meta['stoi'], meta['itos']
    encode = lambda s: [stoi[c] for c in s]
    decode = lambda l: ''.join([itos[i] for i in l])
    start_ids = encode(prompt_tokens)
    x = (torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...])
    number_samples = 1
    # run generation
    with torch.no_grad():
        y = model.generate(x, max_new_tokens, temperature=0.8, top_k=top_k)
        generated.append(decode(y[0].tolist()))
    return generated

In [16]:
generate(model,meta,prompt_tokens=None)

["\nthe son of his folly,\nThat in the brother is as enemy's sin\nHath received note to be to known: if he"]

In [ ]:
# PPO training loop
STEPS = 25
PPO_EPSILON = 0.2  # clipping range
learning_rate = 1e-4

rl_model = copy.deepcopy(model)
rl_model.train()
ref_model = copy.deepcopy(model)
ref_model.eval()

rl_optim = torch.optim.Adam(rl_model.parameters(), lr=learning_rate)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

rl_model = rl_model.to(device)
ref_model = ref_model.to(device)
reward_model = reward_model.to(device)

for step in range(STEPS):
    # Generate trajectories from policy model
    with torch.no_grad():
        # start with empty prompt or a seed
        generated_tokens = generate(rl_model,meta, prompt_tokens=None, max_new_tokens=50, device=device)
        # (1, seq_len)
        obs = generated_tokens[0]  # shape (seq_len,)
        actions = obs[1:]  # all but first token
        
        # Get reward for this trajectory
        # process text 
        #remove new linesince the encoder does not have one
        generated_tokens = generated_tokens[0].replace("\n", "")
        encoded = [ reward_meta_data['stoi'][ch]  for ch in generated_tokens]
        inputs = torch.tensor(encoded, dtype=torch.long).unsqueeze(0)  # Add batch dimension
        inputs = inputs.to(device)

        reward_score = reward_model(inputs)  
        print(reward_score)
        print(type(actions))
        input_actions = torch.tensor(actions)
        rewards = reward_score.expand_as(actions).float()  # (seq_len-1,)
    
    # Forward pass: compute log probs from RL model
    logits_rl = rl_model(generated_tokens)  # (1, seq_len, vocab_size)
    log_probs_rl = torch.nn.functional.log_softmax(logits_rl[0, :-1, :], dim=-1)  
    log_probs = log_probs_rl[torch.arange(len(actions), device=device), actions] 
    
    # Compute reference log probs (no gradient)
    with torch.no_grad():
        logits_ref = ref_model(generated_tokens) 
        log_probs_ref = torch.nn.functional.log_softmax(logits_ref[0, :-1, :], dim=-1)  
        log_probs_ref = log_probs_ref[torch.arange(len(actions), device=device), actions]  
    
    # PPO loss computation
    ratio = torch.exp(log_probs - log_probs_ref)  
    
    ppo_loss1 = rewards * ratio
    ppo_loss2 = rewards * torch.clamp(ratio, 1.0 - PPO_EPSILON, 1.0 + PPO_EPSILON)
    
    ppo_loss = -torch.min(ppo_loss1, ppo_loss2).mean()
    
    # Optional: add log-likelihood loss for behavior cloning
    log_lik_loss = -(rewards * log_probs).mean()
    
    total_loss = ppo_loss + 0.01 * log_lik_loss  # weight log-lik loss lightly
    
    # Backward pass
    rl_optim.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(rl_model.parameters(), 1.0)
    rl_optim.step()
    
    # Logging
    if step % 5 == 0:
        print(f"step = {step:2d} | ppo_loss={ppo_loss.item():.4f} | log_lik_loss={log_lik_loss.item():.4f} | reward={reward_score.item():.4f}")

tensor([1.6721], device='cuda:0')
<class 'str'>


TypeError: expand_as(): argument 'other' (position 1) must be Tensor, not str

In [ ]:
# starting with the reward model train a policy gradient 

# basic policy gradient reinforcement learning

#vanilla policy gradient
rl_model = copy.deepcopy(model)
rl_model.train(False)

ref_model = copy.deepcopy(model)
rl_optim = torch.optim.Adam(rl_model.parameters())
rl_optim.zero_grad()


In [22]:
STEPS = 25

for step in range(STEPS):

    with torch.no_grad():
        traj = generate(rl_model, meta, max_new_tokens=50, device=device)

    
    traj = torch.tensor(traj, dtype=torch.long, device=device)

    obs = traj[:-1]
    actions = traj[1:]

    text = decode(meta, traj.tolist()).replace("\n", "")
    encoded = [reward_meta_data["stoi"].get(ch, 0) for ch in text]
    reward_input = torch.tensor([encoded], dtype=torch.long, device=device)
    reward_score = reward_model(reward_input).squeeze()
    rewards = reward_score.repeat(actions.shape[0])

    logits = rl_model(obs.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)
    logprobs_actions = logprobs[torch.arange(len(actions)), actions]
    loss = -(rewards * logprobs_actions).mean()

    with torch.no_grad():
        names = generate(rl_model, device=device, seed=42 + step)
        print(f"step={step:2d} loss={loss.item():.4f} reward={reward_score.item():.4f}")
        print(",".join(map(str, names)))

    rl_optim.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(rl_model.parameters(), 1.0)
    rl_optim.step()


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
